In [ ]:
import cudf
import numpy as np
import plotly.graph_objects as go
from tqdm import tqdm
import os
from scipy.stats import gaussian_kde
import altair as alt
%load_ext cudf.pandas
import pandas as pd
import pygwalker as pyg
import polars as pl
import matplotlib.pyplot as plt
import mplcursors
%matplotlib notebook
# Fonction pour charger les données parquet
def load_parquet(stock, date):
    file_path = f'/home/janis/3A/EA/HFT_QR_RL/data/smash4/DB_MBP_10/{stock}/{stock}_{date}.parquet'
    return cudf.read_parquet(file_path)

# Spécifier les dates et stocks
# Get all dates from the parquet files in the folder
folder_path = '/home/janis/3A/EA/HFT_QR_RL/data/smash4/DB_MBP_10/AAL'
dates = [f.split('_')[1].split('.')[0] for f in os.listdir(folder_path) if f.endswith('.parquet')]
dates.sort() # Sort dates chronologically
stocks = ["CXW"]


In [15]:
# Function to load parquet data
def load_parquet(stock, date):
    file_path = f'/home/janis/3A/EA/HFT_QR_RL/data/smash4/DB_MBP_10/{stock}/{stock}_{date}.parquet'
    return pl.read_parquet(file_path)

# Specify dates and stocks
dates = ["2024-07-22"]
stocks = ["CSX"]

# Load data for each stock and date
data_dict = {}
for stock in stocks:
    data_dict[stock] = {}
    for date in dates:
        data_dict[stock][date] = load_parquet(stock, date).sample(fraction=1, seed=1)


In [ ]:

# Calculate statistics
stats_dict = {}
for stock in stocks:
    stats_dict[stock] = {}
    for date in dates:
        df = data_dict[stock][date]
        
        mid_price = (df['bid_px_00'] + df['ask_px_00']) / 2
        spread = df['ask_px_00'] - df['bid_px_00']
        
        stats = {
            'mean_mid_price': float(mid_price.mean()),
            'std_mid_price': float(mid_price.std()),
            'min_mid_price': float(mid_price.min()),
            'max_mid_price': float(mid_price.max()),
            'mean_spread': float(spread.mean()),
            'std_spread': float(spread.std()),
            'min_spread': float(spread.min()),
            'max_spread': float(spread.max()),
            'total_volume_bid': int(df['bid_ct_00'].sum()),
            'total_volume_ask': int(df['ask_ct_00'].sum()),
            'num_quotes': len(df),
        }
        
        stats_dict[stock][date] = stats

# Print statistics
for stock in stats_dict:
    print(f"\nStatistics for {stock}:")
    for date in stats_dict[stock]:
        print(f"\nDate: {date}")
        for metric, value in stats_dict[stock][date].items():
            if 'price' in metric:
                print(f"{metric}: ${value:.4f}")
            elif 'spread' in metric:
                print(f"{metric}: ${value:.6f}")
            else:
                print(f"{metric}: {value:,}")


In [ ]:
# Create visualization for each stock and date
for date in dates:
    for stock in tqdm(stocks, desc="Processing stocks"):
        # Get data and calculate mid price
        df = data_dict[stock][date].sort('ts_event')
        df = df.with_columns([
            ((pl.col('bid_px_00') + pl.col('ask_px_00')) / 2).alias('mid_price')
        ])
        
        # Convert to pandas for plotting
        pdf = df.to_pandas()
        
        # Create figure
        fig = go.Figure()
        
        # Add mid price line
        fig.add_trace(go.Scatter(
            x=pdf['ts_event'],
            y=pdf['mid_price'],
            mode='lines',
            name='Mid Price',
            line=dict(color='black', width=1)
        ))
        
        # Add best bid/ask scatter with volume-based size
        fig.add_trace(go.Scatter(
            x=pdf['ts_event'],
            y=pdf['bid_px_00'],
            mode='markers',
            name='Best Bid',
            marker=dict(
                size=pdf['bid_sz_00']/10,
                color='green',
                opacity=0.3
            )
        ))
        
        fig.add_trace(go.Scatter(
            x=pdf['ts_event'],
            y=pdf['ask_px_00'],
            mode='markers',
            name='Best Ask',
            marker=dict(
                size=pdf['ask_sz_00']/10,
                color='red',
                opacity=0.3
            )
        ))
        
        # Add second best bid/ask lines
        fig.add_trace(go.Scatter(
            x=pdf['ts_event'],
            y=pdf['bid_px_01'],
            mode='lines',
            name='Second Best Bid',
            line=dict(color='lightgreen', width=1),
            opacity=0.3
        ))
        
        fig.add_trace(go.Scatter(
            x=pdf['ts_event'],
            y=pdf['ask_px_01'],
            mode='lines',
            name='Second Best Ask',
            line=dict(color='pink', width=1),
            opacity=0.3
        ))
        
        # Add trades
        trades = pdf[pdf['rtype'] == 2]
        fig.add_trace(go.Scatter(
            x=trades['ts_event'],
            y=trades['mid_price'],
            mode='markers',
            name='Trades',
            marker=dict(
                size=10,
                color='black',
                opacity=0.7
            )
        ))
        
        # Update layout
        fig.update_layout(
            title=f"Order Book Visualization for {stock} on {date}",
            xaxis_title="Time",
            yaxis_title="Price",
            showlegend=True,
            template="plotly_white"
        )
        
        fig.show()
